In [81]:
import pandas as pd
import numpy as np

df = pd.read_csv("./DirtyData.csv")
df.head(20)

,first_name,last_name,email,gender,income,tax_15
0,Alverta,Colkett,acolkett0@cocolog-nifty.com,Female,123072.34,NaN
1,Nichole,Brandassi,nbrandassi1@mail.ru,Bigender,NaN,24115.00
2,Ruperto,Chaddock,rchaddock2@mail.ru,Male,62524.77,9378.72
3,Lula,Sorrill,lsorrill3@hatena.ne.jp,Female,48000.77,7200.12
4,Blondell,Benard,bbenard4@admin.ch,Female,88638.49,13295.77
5,Jacquelyn,Fawdry,jfawdry5@addthis.com,Female,NaN,14493.00
6,Kurt,Dugue,kdugue6@istockphoto.com,Male,105041.02,15756.15
7,Cordelie,Bloxsom,cbloxsom7@state.gov,Woman,236589.68,35488.45
8,Katinka,Renzo,krenzo8@thetimes.co.uk,Woman,194301.63,29145.24
9,Ketty,Pakeman,kpakeman9@chicagotribune.com,Female,239219.05,NaN


### Step 1: Inspect the data before doing anything like guessing

In [82]:
print(df.shape)
print(df.isna().sum())
print(f"\nNegative value: {(df["income"] < 0).sum()}")
df["gender"].value_counts()

(1000, 6)
first_name      0
last_name       0
email           0
gender          0
income        115
tax_15         28
dtype: int64

Negative value: 86


gender
Female         469
Male           409
Non-binary      21
Genderfluid     19
Polygender      18
Genderqueer     17
Bigender        16
Agender         13
Man             10
Woman            3
Men              3
Women            2
Name: count, dtype: int64

### Step 2: Handle missing values by reconstruction (not guessing)

In [83]:
#tax_value = 15/100 * income
#income = 100/15 * tax_value
df["income"] = np.where(df["income"].isna() & df["tax_15"].notna(), df["tax_15"] * (100/15), df["income"])
df["tax_15"] = np.where(df["tax_15"].isna() & df["income"].notna(), df["income"] * [15/100], df["tax_15"])
df.head(20)

,first_name,last_name,email,gender,income,tax_15
0,Alverta,Colkett,acolkett0@cocolog-nifty.com,Female,123072.340000,18460.8510
1,Nichole,Brandassi,nbrandassi1@mail.ru,Bigender,160766.666667,24115.0000
2,Ruperto,Chaddock,rchaddock2@mail.ru,Male,62524.770000,9378.7200
3,Lula,Sorrill,lsorrill3@hatena.ne.jp,Female,48000.770000,7200.1200
4,Blondell,Benard,bbenard4@admin.ch,Female,88638.490000,13295.7700
5,Jacquelyn,Fawdry,jfawdry5@addthis.com,Female,96620.000000,14493.0000
6,Kurt,Dugue,kdugue6@istockphoto.com,Male,105041.020000,15756.1500
7,Cordelie,Bloxsom,cbloxsom7@state.gov,Woman,236589.680000,35488.4500
8,Katinka,Renzo,krenzo8@thetimes.co.uk,Woman,194301.630000,29145.2400
9,Ketty,Pakeman,kpakeman9@chicagotribune.com,Female,239219.050000,35882.8575


### Step 3: Fix the negative values

In [84]:
df["income"] = df["income"].abs()
df["tax_15"] = df["tax_15"].abs()

### Step 4: Standarize `gender` with Judgement
Category Man, Men, Male, Women, Woman, Female are self-evident. However, the rest of the categories we should not flatten those into Male/Female. If we convert those it would be misrepresentation and destroy our information.

In [85]:
df["gender"].value_counts()

gender_map = {
    "Man": "Male", "Men": "Male",
    "Woman": "Female", "Women": "Female"
}

df["gender"] = df["gender"].replace(gender_map)

df.head(20)
#df["gender"].value_counts()

,first_name,last_name,email,gender,income,tax_15
0,Alverta,Colkett,acolkett0@cocolog-nifty.com,Female,123072.340000,18460.8510
1,Nichole,Brandassi,nbrandassi1@mail.ru,Bigender,160766.666667,24115.0000
2,Ruperto,Chaddock,rchaddock2@mail.ru,Male,62524.770000,9378.7200
3,Lula,Sorrill,lsorrill3@hatena.ne.jp,Female,48000.770000,7200.1200
4,Blondell,Benard,bbenard4@admin.ch,Female,88638.490000,13295.7700
5,Jacquelyn,Fawdry,jfawdry5@addthis.com,Female,96620.000000,14493.0000
6,Kurt,Dugue,kdugue6@istockphoto.com,Male,105041.020000,15756.1500
7,Cordelie,Bloxsom,cbloxsom7@state.gov,Female,236589.680000,35488.4500
8,Katinka,Renzo,krenzo8@thetimes.co.uk,Female,194301.630000,29145.2400
9,Ketty,Pakeman,kpakeman9@chicagotribune.com,Female,239219.050000,35882.8575


### Step 5: Normalize income (min-max)

In [86]:
# (value - min) / range
# range = max_income - min_income

min = df["income"].min()
max = df["income"].max()

income_range = (max - min)
df["income_norm"] = round((df["income"] - min) / income_range, 3)

print("Range for income: ", round(min, 2), round(max,2))

print("Range for inormalized income:", round(df["income_norm"].min(),2), round(df["income_norm"].max(), 2))

df.head(20)

Range for income:  15245.38 326986.67
Range for inormalized income: 0.0 1.0


,first_name,last_name,email,gender,income,tax_15,income_norm
0,Alverta,Colkett,acolkett0@cocolog-nifty.com,Female,123072.340000,18460.8510,0.346
1,Nichole,Brandassi,nbrandassi1@mail.ru,Bigender,160766.666667,24115.0000,0.467
2,Ruperto,Chaddock,rchaddock2@mail.ru,Male,62524.770000,9378.7200,0.152
3,Lula,Sorrill,lsorrill3@hatena.ne.jp,Female,48000.770000,7200.1200,0.105
4,Blondell,Benard,bbenard4@admin.ch,Female,88638.490000,13295.7700,0.235
5,Jacquelyn,Fawdry,jfawdry5@addthis.com,Female,96620.000000,14493.0000,0.261
6,Kurt,Dugue,kdugue6@istockphoto.com,Male,105041.020000,15756.1500,0.288
7,Cordelie,Bloxsom,cbloxsom7@state.gov,Female,236589.680000,35488.4500,0.710
8,Katinka,Renzo,krenzo8@thetimes.co.uk,Female,194301.630000,29145.2400,0.574
9,Ketty,Pakeman,kpakeman9@chicagotribune.com,Female,239219.050000,35882.8575,0.718


### Step 6: Final Check and Save

In [87]:
df.to_csv("./CleanedData.csv", index=False)

1. Print the five-number summary of the cleaned `income` column (min, Q1, median, Q3, Max).
2. How many people are in each gender category after cleaning?

- git add yourfile.ipyb
- git commit - m "your message"
- git push

In [89]:
import numpy as np
df.describe()

,income,tax_15,income_norm
count,1000.000000,1000.000000,1000.000000
mean,140011.671460,21001.751017,0.400214
std,71658.735725,10748.810452,0.229882
min,15245.380000,2286.810000,0.000000
25%,80435.717500,12065.357500,0.209000
50%,139509.475000,20926.425000,0.398500
75%,201023.587500,30153.540000,0.596250
max,326986.666667,49048.000000,1.000000
